# Mini-projet Data Mining : détection automatique des crises d'absence à partir de signaux EEG

**Objectif :** construire une chaîne complète *signal EEG brut → features → modèle de classification → début et fin de chaque crise*.

Ce notebook suit l'ordre de l'énoncé (Parties I à XII). Chaque étape contient :
- 🧠 **Comprendre** : l'idée, expliquée simplement ;
- 💻 le **code** ;
- ✅ **Réponse** : la réponse à la question de l'énoncé, à partir des résultats obtenus.

**Pipeline :** Signal EEG → Exploration → Prétraitement → Segmentation → Extraction des features → Dataset (classe 0/1) → DT / RF / KNN / SVM → Évaluation → Meilleur modèle → Détection temporelle → Début/Fin de chaque crise.

## 0. Installation et imports

Les fichiers `.mat` ont été sauvegardés par MATLAB sous forme de **`timetable`**. `scipy.io.loadmat` ne sait pas lire ce type d'objet, alors on utilise la librairie **`mat-io`**, qui convertit un `timetable` MATLAB directement en `DataFrame` pandas.

Installation, à faire une seule fois dans un terminal :
```
pip install --upgrade mat-io scipy numpy pandas matplotlib seaborn scikit-learn openpyxl
```

Place ce notebook **dans le dossier qui contient les fichiers `.mat` et `Crises absences.xlsx`**. L'exécution complète prend quelques minutes.

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, sosfiltfilt
from matio import load_from_mat            # pip install mat-io

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
pd.set_option("display.max_columns", 30)

# Dossier qui contient les fichiers .mat et l'Excel (par défaut, le dossier du notebook)
DATA_DIR = os.environ.get("EEG_DIR", ".")
EXCEL = os.path.join(DATA_DIR, "Crises absences.xlsx")
RANDOM_STATE = 42

# Partie I : Compréhension et exploration des données

🧠 **Comprendre.** Pour chaque patient, on dispose de deux fichiers :
- `XXXX_0000d.mat` (**d** = *data*) : le signal EEG. Chaque ligne du `timetable` correspond à **1 seconde** et contient, pour chaque canal, un petit tableau de 256 valeurs (les 256 échantillons de cette seconde).
- `XXXX_0000h.mat` (**h** = *header* ou annotations) : les notes du technicien pendant l'examen (« HPN » pour hyperpnée, « YO/YF » pour yeux ouverts/fermés, « ARTEFACT », « absence »…).

On commence par un seul patient : **211104B-D**.

## Question 1 : charger un fichier EEG

In [ ]:
PATIENT = "211104B-D"
fichier_d = os.path.join(DATA_DIR, PATIENT.replace("-", "_") + "_0000d.mat")

tt = load_from_mat(fichier_d)["output_d"]      # timetable MATLAB -> DataFrame pandas
print(type(tt), tt.shape)
tt.iloc[:3, :4]

Chaque cellule contient un tableau de 256 valeurs. On **déplie** ce tableau pour obtenir un DataFrame classique : **une ligne = un échantillon**, avec une colonne `Time` en secondes et une colonne par canal (comme dans le tableau de l'énoncé).

In [ ]:
CANAUX_EEG = ['EEGFp1','EEGFp2','EEGF7','EEGF3','EEGFz','EEGF4','EEGF8','EEGT3','EEGC3',
              'EEGCz','EEGC4','EEGT4','EEGT5','EEGP3','EEGPz','EEGP4','EEGT6','EEGO1','EEGO2']

def charger_eeg(patient, canaux=CANAUX_EEG):
    # Charge le fichier _d.mat d'un patient et renvoie (DataFrame, fs)
    f = os.path.join(DATA_DIR, patient.replace("-", "_") + "_0000d.mat")
    tt = load_from_mat(f)["output_d"]
    fs = np.ravel(tt.iloc[0, 0]).size                      # nb d'échantillons par seconde
    data = {c: np.concatenate([np.ravel(v) for v in tt[c]]) for c in canaux}
    df = pd.DataFrame(data).astype("float32")
    df.insert(0, "Time", np.arange(len(df)) / fs)
    return df, fs

df, fs = charger_eeg(PATIENT)
canaux_dispo = load_from_mat(fichier_d)["output_d"].columns.tolist()

print("Canaux présents dans le fichier :", canaux_dispo)
print(f"Nombre de lignes (échantillons) : {df.shape[0]:,}")
print(f"Nombre de variables             : {df.shape[1]}  (Time + {df.shape[1]-1} canaux EEG)")
print(f"Fréquence d'échantillonnage     : {fs} Hz")
print(f"Durée totale                    : {df.shape[0]/fs:.0f} s  = {df.shape[0]/fs/60:.1f} min")
print(f"Valeurs manquantes              : {int(df.isna().sum().sum())}")
df.head()

✅ **Réponse Q1.**
- **Fréquence d'échantillonnage : 256 Hz**, c'est-à-dire 256 mesures par seconde et par canal (identique pour tous les patients).
- Le fichier contient **19 canaux EEG** placés selon le système international 10-20 (Fp = frontal polaire, F = frontal, C = central, T = temporal, P = pariétal, O = occipital ; impair = gauche, pair = droite, z = ligne médiane). Il contient aussi des canaux non-EEG (ECG = cœur, SLI = stimulation lumineuse, EMG = muscles) que l'on **écarte**.
- **Aucune valeur manquante**, et l'enregistrement est continu (pas de trou).
- Le nombre de lignes, de variables et la durée sont affichés ci-dessus. L'unité de l'amplitude est le **microvolt (µV)**.

In [ ]:
# Statistiques descriptives rapides de quelques canaux
df[["EEGFp1", "EEGC3", "EEGO1", "EEGT4"]].describe().round(1)

## Prétraitement : filtrage

🧠 **Comprendre.** Le signal brut contient :
- une **dérive lente** (< 0.5 Hz), due à la transpiration et aux mouvements des électrodes ;
- du **bruit haute fréquence** (> 40 Hz), dû au secteur électrique à 50 Hz et aux muscles.

L'information utile de l'EEG se situe entre **0.5 et 40 Hz**. On applique donc un **filtre passe-bande Butterworth 0.5–40 Hz**. On utilise `sosfiltfilt`, qui filtre dans les deux sens et ne décale donc pas le signal dans le temps.

In [ ]:
def filtrer(X, fs, low=0.5, high=40, ordre=4):
    sos = butter(ordre, [low, high], btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, X, axis=0).astype("float32")

X_brut = df[CANAUX_EEG].values
X_filt = filtrer(X_brut, fs)

t = df["Time"].values
sl = (t >= 100) & (t < 110)
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t[sl], X_brut[sl, 0], label="brut", alpha=.6)
ax.plot(t[sl], X_filt[sl, 0], label="filtré 0.5–40 Hz")
ax.set(title="Canal Fp1 : effet du filtrage", xlabel="Temps (s)", ylabel="Amplitude (µV)")
ax.legend(); plt.show()

## Question 2 : représenter le signal EEG en fonction du temps

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(t, X_filt[:, CANAUX_EEG.index("EEGF3")], lw=.4)
ax.set(title=f"Patient {PATIENT} : signal EEG (canal F3) sur tout l'enregistrement",
       xlabel="Temps (s)", ylabel="Amplitude (µV)")
plt.show()

Pour bien voir l'EEG, on affiche plusieurs canaux empilés (comme sur l'écran d'un neurologue). Voici une fenêtre de 20 s autour d'une crise (les instants de crise sont calculés dans la Partie II).

In [ ]:
def plot_multicanal(X, fs, t0, t1, titre, canaux=CANAUX_EEG, crises=None, ecart=250):
    i0, i1 = int(t0*fs), int(t1*fs)
    tt_ = np.arange(i0, i1) / fs
    fig, ax = plt.subplots(figsize=(14, 7))
    for k, c in enumerate(canaux):
        ax.plot(tt_, X[i0:i1, CANAUX_EEG.index(c)] - k*ecart, lw=.6, color="k")
    if crises is not None:
        for d, f in crises:
            if f > t0 and d < t1:
                ax.axvspan(max(d, t0), min(f, t1), color="red", alpha=.15)
    ax.set_yticks([-k*ecart for k in range(len(canaux))], [c.replace("EEG", "") for c in canaux])
    ax.set(title=titre, xlabel="Temps (s)"); ax.set_xlim(t0, t1)
    plt.show()

# Partie II : Exploitation des annotations

🧠 **Comprendre.** Le fichier Excel `Crises absences.xlsx` donne, pour chaque patient, `DebCE1`, `FinCE1`, `DebCE2`… Ces valeurs (par exemple `0.4617`) sont des **heures au format Excel**, c'est-à-dire une **fraction de journée** :

$$ \text{secondes depuis minuit} = \text{valeur} \times 86400 \qquad (0.4617 \times 86400 = 39\,893 \text{ s} = 11\text{h}04\text{min}53\text{s}) $$

⚠️ **Problème :** ce sont des **heures d'horloge** (11:04:53), alors que le signal EEG est indexé depuis **0 s** (début de l'enregistrement). L'heure de début de l'enregistrement n'est stockée nulle part dans les fichiers.

💡 **Solution : recalage automatique.** Pendant une crise d'absence, l'EEG montre des **pointes-ondes à ~3 Hz** très amples. On calcule donc, seconde par seconde, la **puissance du signal entre 2.5 et 4.5 Hz** (moyennée sur les 19 canaux). On teste ensuite toutes les heures de début possibles et on garde celle qui place les crises de l'Excel sur les zones où cette puissance est maximale.

On **vérifie** ensuite avec le fichier `_h.mat` : quand le technicien a écrit « absence » ou « crise », cette note doit tomber sur une crise recalée. Le recalage est confirmé sur 10 patients, et l'écart est de quelques secondes (le technicien écrit sa note pendant la crise).

In [ ]:
# ---- 1) Lecture de l'Excel ----
xl = pd.read_excel(EXCEL)
xl = xl.rename(columns={xl.columns[0]: "ID"})
xl["ID"] = xl["ID"].astype(str).str.strip()
xl = xl[xl["ID"].str.match(r"\d{6}[AB]-[A-Z]")]
print("Lignes dupliquées :", xl["ID"][xl["ID"].duplicated()].tolist())
xl = xl.drop_duplicates("ID").reset_index(drop=True)

COL_DEB = [c for c in xl.columns if c.startswith("Deb")]
COL_FIN = [c for c in xl.columns if c.startswith("Fin")]

def crises_excel(row):
    # Liste des (début, fin) en secondes depuis minuit
    out = []
    for d, f in zip(COL_DEB, COL_FIN):
        if pd.notna(row[d]) and pd.notna(row[f]):
            out.append((row[d] * 86400, row[f] * 86400))
    return out

xl[["ID", "Sexe", "Age", "NombreCE"] + COL_DEB[:2] + COL_FIN[:2]].head()

In [ ]:
# ---- 2) Contrôle qualité des annotations ----
for _, r in xl.iterrows():
    cr = crises_excel(r)
    fichier = os.path.join(DATA_DIR, r.ID.replace("-", "_") + "_0000d.mat")
    pb = []
    if not os.path.exists(fichier): pb.append("fichier .mat absent")
    if len(cr) == 0: pb.append("aucune crise annotée")
    if any(f - d > 120 for d, f in cr): pb.append(f"crise de {max(f-d for d,f in cr)/3600:.1f} h (erreur de saisie)")
    if pd.notna(r.NombreCE) and len(cr) != r.NombreCE: pb.append(f"NombreCE={r.NombreCE} mais {len(cr)} crises")
    if pb: print(f"{r.ID:10s} -> {' ; '.join(pb)}")

**Décisions de nettoyage :**
- `200625A-F` : aucune crise annotée et seulement 8 canaux EEG → **exclu** ;
- `211222B-G` : pas de fichier `.mat` → **exclu** ;
- `210406A-F` : une « crise » de 3 heures, ce qui est une erreur de saisie évidente → **exclu** ;
- `230515B-G` apparaît deux fois → on garde la première ligne ;
- `210914B-A` : `NombreCE = 6` alors que 11 crises sont saisies → on se fie aux dates saisies (11 crises).

On garde donc **20 patients**.

In [ ]:
def score_pointes_ondes(X, fs):
    # Pour chaque seconde : log-puissance 2.5-4.5 Hz moyennée sur les canaux, puis normalisée
    n = X.shape[0] // fs
    seg = X[: n*fs].reshape(n, fs, X.shape[1])
    spec = np.abs(np.fft.rfft(seg, axis=1))**2
    f = np.fft.rfftfreq(fs, 1/fs)
    p = spec[:, (f >= 2.5) & (f <= 4.5), :].sum(1)
    s = np.log(p + 1e-6).mean(1)
    return (s - np.median(s)) / s.std()

def annotations_h(patient):
    h = load_from_mat(os.path.join(DATA_DIR, patient.replace("-", "_") + "_0000h.mat"))["output_h"]
    lab = h["Annotations"].astype(str)
    return pd.DataFrame({"t": h.index.total_seconds(), "texte": lab.values})

def recaler(crises_abs, score, ancres=()):
    # Cherche l'heure de début t0 (en s depuis minuit) qui aligne au mieux les crises sur le score.
    # Bonus si les notes 'absence/crise' du technicien tombent dans les crises.
    n = len(score)
    premier = min(d for d, f in crises_abs); dernier = max(f for d, f in crises_abs)
    meilleur = (-np.inf, None)
    for t0 in np.arange(np.floor(dernier - n), np.ceil(premier) + 1):
        idx = np.concatenate([np.arange(round(d - t0), round(f - t0) + 1) for d, f in crises_abs])
        idx = idx[(idx >= 0) & (idx < n)]
        s = score[idx].mean()
        if len(ancres):
            s += 2 * np.mean([any(d-t0-5 <= a <= f-t0+15 for d, f in crises_abs) for a in ancres])
        if s > meilleur[0]: meilleur = (s, t0)
    return meilleur[1]

In [ ]:
# ---- Recalage du patient étudié ----
row = xl[xl.ID == PATIENT].iloc[0]
score = score_pointes_ondes(X_filt, fs)
h = annotations_h(PATIENT)
ancres = h.t[h.texte.str.lower().str.contains("absence|absnce|crise")].values
t0 = recaler(crises_excel(row), score, ancres)
crises = [(d - t0, f - t0) for d, f in crises_excel(row)]
print("Heure de début de l'enregistrement estimée :", pd.to_timedelta(t0, unit="s"))

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(np.arange(len(score)), score, lw=.7, label="score pointes-ondes (2.5–4.5 Hz)")
for d, f in crises: ax.axvspan(d, f, color="red", alpha=.3)
ax.set(title=f"{PATIENT} : le score est maximal pendant les crises recalées (en rouge)",
       xlabel="Temps (s)", ylabel="score (z)"); ax.legend(); plt.show()

## Question 3 : début, fin et nombre total de crises

In [ ]:
tab_crises = pd.DataFrame(crises, columns=["Début (s)", "Fin (s)"])
tab_crises.index = np.arange(1, len(tab_crises) + 1); tab_crises.index.name = "Crise"
tab_crises["Durée (s)"] = tab_crises["Fin (s)"] - tab_crises["Début (s)"]
print("Nombre total de crises :", len(tab_crises))
tab_crises.round(1)

In [ ]:
d0, f0 = crises[0]
plot_multicanal(X_filt, fs, d0 - 8, f0 + 8, f"{PATIENT} : crise n°1 (zone rouge), 19 canaux", crises=crises)

✅ **Réponse Q2 : que remarque-t-on visuellement pendant les crises ?**
- L'**amplitude augmente fortement** et **brusquement**, sur **tous les canaux en même temps**. C'est une crise **généralisée**, qui touche les deux hémisphères.
- Le signal devient **très rythmique et régulier** : on voit des complexes **pointe-onde** (une pointe rapide suivie d'une onde lente) qui se répètent environ **3 fois par seconde**. C'est la signature typique de l'absence.
- Le début et la fin sont **nets** : le tracé redevient normal immédiatement après la crise.

## Question 4 : statistiques des durées (patient étudié, puis tous les patients)

In [ ]:
def stats_crises(cr):
    dur = np.array([f - d for d, f in cr])
    inter = np.array([cr[i+1][0] - cr[i][1] for i in range(len(cr)-1)])
    return {"nb": len(cr), "durée moy": dur.mean(), "durée min": dur.min(), "durée max": dur.max(),
            "intervalle moy": inter.mean() if len(inter) else np.nan}
pd.Series(stats_crises(crises)).round(1)

On applique maintenant le même traitement (chargement, filtrage, recalage) aux 19 patients. On en profite pour **conserver le signal filtré** de chaque patient, car il servira pour la segmentation.

In [ ]:
EXCLUS = {"200625A-F", "211222B-G", "210406A-F"}
SIGNAUX, CRISES, INFOS = {}, {}, []
for _, r in xl.iterrows():
    if r.ID in EXCLUS: continue
    dfp, fs_p = charger_eeg(r.ID)
    Xp = filtrer(dfp[CANAUX_EEG].values, fs_p)
    sc = score_pointes_ondes(Xp, fs_p)
    hp = annotations_h(r.ID)
    anc = hp.t[hp.texte.str.lower().str.contains("absence|absnce|crise")].values
    t0p = recaler(crises_excel(r), sc, anc)
    SIGNAUX[r.ID] = Xp
    CRISES[r.ID] = [(d - t0p, f - t0p) for d, f in crises_excel(r)]
    INFOS.append({"ID": r.ID, "Sexe": r.Sexe, "Age": r.Age, "durée (min)": len(Xp)/fs_p/60,
                  "début enreg.": str(pd.to_timedelta(round(t0p), unit="s"))[7:],
                  "notes 'absence' du technicien": len(anc), **stats_crises(CRISES[r.ID])})
infos = pd.DataFrame(INFOS).set_index("ID")
infos.round(1)

In [ ]:
toutes = pd.DataFrame([(p, d, f, f-d) for p, cr in CRISES.items() for d, f in cr],
                      columns=["patient", "debut", "fin", "duree"])
print(f"Nombre total de crises : {len(toutes)} chez {toutes.patient.nunique()} patients")
print(f"Durée moyenne  : {toutes.duree.mean():.1f} s")
print(f"Durée médiane  : {toutes.duree.median():.1f} s")
print(f"Durée min/max  : {toutes.duree.min():.1f} s / {toutes.duree.max():.1f} s")
print(f"Intervalle moyen entre deux crises : {infos['intervalle moy'].mean():.0f} s")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(toutes.duree, bins=30, color="tab:red", alpha=.8, edgecolor="w")
ax.set(title="Distribution de la durée des crises (tous patients)", xlabel="Durée (s)", ylabel="Nombre de crises")
plt.show()

✅ **Réponse : les crises ont-elles toutes une durée similaire ?**
**Non.** Sur les **92 crises**, la durée **médiane est de 10 s** et la moyenne de 12.7 s. La plupart durent **entre 3 et 20 secondes**, ce qui est typique des absences, mais la dispersion est grande : de **2 s** à **60 s**. La durée varie **d'un patient à l'autre** et **au sein d'un même patient**.

Conséquences pour la suite :
1. une fenêtre de 2 s est assez courte pour « voir » même les crises brèves ;
2. les crises très courtes (< 2 s) risquent d'être ratées, car une fenêtre n'est de classe 1 que si au moins 50 % de ses échantillons sont en crise.

# Partie III : Construction de la variable cible

🧠 **Comprendre.** On transforme le problème en **classification binaire**. Chaque échantillon reçoit `Class = 1` s'il se trouve dans un intervalle `[début crise, fin crise]`, et `Class = 0` sinon.

## Question 5 : création de la colonne `Class`

In [ ]:
def vecteur_classe(n_echantillons, fs, crises):
    t = np.arange(n_echantillons) / fs
    y = np.zeros(n_echantillons, dtype=int)
    for d, f in crises:
        y[(t >= d) & (t <= f)] = 1
    return y

df["Class"] = vecteur_classe(len(df), fs, crises)
df[["Time", "EEGFp1", "EEGC3", "Class"]].iloc[int(crises[0][0]*fs)-3 : int(crises[0][0]*fs)+3]

## Question 6 : afficher le signal en distinguant les périodes hors crise et pendant une crise

In [ ]:
sig = X_filt[:, CANAUX_EEG.index("EEGF3")]
y = df["Class"].values
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(t, np.where(y == 0, sig, np.nan), lw=.4, color="tab:blue", label="hors crise (0)")
ax.plot(t, np.where(y == 1, sig, np.nan), lw=.4, color="tab:red", label="crise (1)")
ax.set(title=f"{PATIENT} : canal F3 coloré selon la classe", xlabel="Temps (s)", ylabel="Amplitude (µV)")
ax.legend(loc="upper right"); plt.show()

pct = df["Class"].value_counts(normalize=True).sort_index() * 100
print(f"Patient {PATIENT} : classe 0 = {pct[0]:.2f} %   classe 1 = {pct[1]:.2f} %")
tot_crise = sum((vecteur_classe(len(X), fs, CRISES[p]) == 1).sum() for p, X in SIGNAUX.items())
tot = sum(len(X) for X in SIGNAUX.values())
print(f"Tous patients    : classe 0 = {100-100*tot_crise/tot:.2f} %   classe 1 = {100*tot_crise/tot:.2f} %")

✅ **Réponse : le déséquilibre est-il important ?**
**Oui, très.** Seulement **3.3 %** des échantillons du patient étudié sont en crise (**3.9 %** sur l'ensemble des patients), contre environ 96 % hors crise. C'est un problème **fortement déséquilibré**, qui a deux conséquences :
- un modèle « paresseux » qui répond toujours 0 aurait déjà environ 96 % d'accuracy (voir Question 13) ;
- il faudra regarder le **Recall**, la **Precision** et le **F1** de la classe 1, et éventuellement **rééquilibrer** les classes (Partie XII).

# Partie IV : Segmentation du signal

🧠 **Comprendre.** Un échantillon isolé (1/256 s) ne contient aucune information sur le **rythme** du signal. On découpe donc le signal en **fenêtres de 2 s** (512 échantillons), avec un **recouvrement de 50 %** : on avance de 1 s (256 échantillons) à chaque fois.

```
fenêtre 1 : [0 s – 2 s]
fenêtre 2 :      [1 s – 3 s]
fenêtre 3 :           [2 s – 4 s] ...
```
Le recouvrement donne plus d'exemples et évite de couper une crise « au mauvais endroit ».

**Classe d'une fenêtre :** 1 si au moins 50 % de ses échantillons sont en crise, 0 sinon.

## Question 7 : découpage en fenêtres

In [ ]:
WIN_S, OVERLAP = 2, 0.5
WIN  = int(WIN_S * fs)              # 512 échantillons
STEP = int(WIN * (1 - OVERLAP))     # 256 échantillons

def segmenter(X, y, fs, win=WIN, step=STEP):
    # Renvoie (fenêtres [n, win, canaux], classes [n], temps de début [n])
    starts = np.arange(0, len(X) - win + 1, step)
    fen = np.stack([X[s:s+win] for s in starts])
    frac_crise = np.array([y[s:s+win].mean() for s in starts])
    return fen, (frac_crise >= 0.5).astype(int), starts / fs

fen, yw, tw = segmenter(X_filt, df["Class"].values, fs)
print("Forme du tableau de fenêtres :", fen.shape, "(fenêtres, échantillons, canaux)")
print("Fenêtres classe 0 :", (yw == 0).sum(), " | classe 1 :", (yw == 1).sum())

# Parties V et VI : Extraction des caractéristiques

🧠 **Comprendre.** On résume chaque fenêtre (512 × 19 valeurs) par quelques nombres, les **features**. Chaque feature est calculée **sur chaque canal**, puis **moyennée sur les 19 canaux** (l'absence est généralisée, donc tous les canaux réagissent).

**Features temporelles** (pour un canal $x_1 \dots x_N$, avec $N = 512$) :

| Feature | Formule | Intuition |
|---|---|---|
| Mean | $\bar x = \frac1N\sum x_i$ | niveau moyen (≈ 0 après filtrage) |
| Std / Variance | $\sigma^2 = \frac1N\sum (x_i-\bar x)^2$ | dispersion : **forte pendant une crise** |
| Min / Max | | pics négatifs et positifs |
| Amplitude | Max − Min | crête à crête |
| RMS | $\sqrt{\frac1N\sum x_i^2}$ | « taille » efficace du signal |
| Energy | $\sum x_i^2$ | énergie totale de la fenêtre |

**Features fréquentielles (FFT).** La transformée de Fourier décompose le signal en une somme de sinusoïdes. La puissance $|X(f)|^2$ dit « combien » il y a de chaque fréquence. On additionne cette puissance dans les bandes cliniques :

| Bande | Fréquences | Remarque |
|---|---|---|
| Delta | 0.5–4 Hz | contient les pointes-ondes à 3 Hz, donc **↑↑ pendant une absence** |
| Theta | 4–8 Hz | harmoniques des pointes |
| Alpha | 8–13 Hz | rythme de repos, yeux fermés |
| Beta | 13–30 Hz | activité rapide, éveil |

## Partie VI : visualisation temporelle et fréquentielle d'une fenêtre

In [ ]:
i_crise = np.where(yw == 1)[0][len(np.where(yw == 1)[0]) // 2]
i_normal = np.where(yw == 0)[0][100]
c = CANAUX_EEG.index("EEGF3")
freqs = np.fft.rfftfreq(WIN, 1/fs)

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
for col, (i, nom, couleur) in enumerate([(i_normal, "hors crise", "tab:blue"), (i_crise, "pendant crise", "tab:red")]):
    x = fen[i, :, c]
    axes[0, col].plot(np.arange(WIN)/fs, x, color=couleur)
    axes[0, col].set(title=f"Domaine temporel : {nom}", xlabel="Temps (s)", ylabel="µV")
    P = np.abs(np.fft.rfft(x * np.hanning(WIN)))**2
    axes[1, col].plot(freqs, P, color=couleur)
    axes[1, col].set(title=f"Spectre (FFT) : {nom}", xlabel="Fréquence (Hz)", ylabel="Puissance", xlim=(0, 30))
    for b in [4, 8, 13]: axes[1, col].axvline(b, ls="--", c="gray", lw=.8)
axes[0, 1].sharey(axes[0, 0])
plt.tight_layout(); plt.show()

👉 Pendant la crise, le spectre a un **pic énorme autour de 3 Hz** (bande Delta), accompagné de ses **harmoniques** (6, 9 Hz…). Hors crise, la puissance est faible et répartie. C'est exactement ce que les features de puissance par bande vont capturer.

## Questions 8 et 9 : calcul des features pour toutes les fenêtres de tous les patients

In [ ]:
BANDES = {"Delta": (0.5, 4), "Theta": (4, 8), "Alpha": (8, 13), "Beta": (13, 30)}
HANN = np.hanning(WIN)

def extraire_features(fen, fs):
    # fen : [n_fenetres, WIN, n_canaux] -> DataFrame de features (moyenne sur les canaux)
    F = {}
    F["Mean"]      = fen.mean(1)
    F["Std"]       = fen.std(1)
    F["Variance"]  = fen.var(1)
    F["Min"]       = fen.min(1)
    F["Max"]       = fen.max(1)
    F["Amplitude"] = F["Max"] - F["Min"]
    F["RMS"]       = np.sqrt((fen**2).mean(1))
    F["Energy"]    = (fen.astype("float64")**2).sum(1)
    # --- FFT ---
    spec = np.abs(np.fft.rfft(fen * HANN[None, :, None], axis=1))**2 / WIN
    f = np.fft.rfftfreq(fen.shape[1], 1/fs)
    for nom, (lo, hi) in BANDES.items():
        F["P_" + nom] = spec[:, (f >= lo) & (f < hi), :].sum(1)
    return pd.DataFrame({k: v.mean(1) for k, v in F.items()})   # moyenne sur les 19 canaux

lignes = []
for p, Xp in SIGNAUX.items():
    yp = vecteur_classe(len(Xp), fs, CRISES[p])
    fen_p, yw_p, tw_p = segmenter(Xp, yp, fs)
    feat = extraire_features(fen_p, fs)
    feat.insert(0, "Patient", p); feat.insert(1, "t_debut", tw_p)
    feat["Class"] = yw_p
    lignes.append(feat)

dataset = pd.concat(lignes, ignore_index=True)
dataset.insert(0, "Window", np.arange(1, len(dataset) + 1))
FEATURES = ["Mean","Std","Variance","Min","Max","Amplitude","RMS","Energy","P_Delta","P_Theta","P_Alpha","P_Beta"]

dataset.to_csv(os.path.join(DATA_DIR, "dataset_fenetres_eeg.csv"), index=False)
print("Nombre total de fenêtres :", len(dataset))
print("Fenêtres classe 0        :", (dataset.Class == 0).sum())
print("Fenêtres classe 1        :", (dataset.Class == 1).sum(), f"({100*dataset.Class.mean():.2f} %)")
dataset.head()

✅ **Réponse Q8.** Le dataset final a **une ligne par fenêtre de 2 s**, avec 12 features (8 temporelles et 4 fréquentielles), la classe et, pour pouvoir revenir dans le temps, le patient et l'instant de début de la fenêtre. On obtient **29 966 fenêtres** : **28 756 de classe 0** et **1 210 de classe 1** (4.0 %). Le dataset est aussi sauvegardé dans `dataset_fenetres_eeg.csv`.

# Partie VII : Analyse exploratoire

## Question 10 : comparaison des features entre les deux classes

In [ ]:
comp = dataset.groupby("Class")[FEATURES].mean().T
comp.columns = ["Classe 0 (hors crise)", "Classe 1 (crise)"]
comp["Ratio crise / hors crise"] = comp.iloc[:, 1] / comp.iloc[:, 0]
print("MOYENNES par classe")
display(comp.round(2))

# La moyenne est très sensible aux valeurs extrêmes (artefacts) : on regarde aussi la médiane
med = dataset.groupby("Class")[FEATURES].median().T
med.columns = ["Classe 0 (hors crise)", "Classe 1 (crise)"]
med["Ratio crise / hors crise"] = med.iloc[:, 1] / med.iloc[:, 0]
print("MÉDIANES par classe")
med.round(2)

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for ax, f in zip(axes.ravel(), FEATURES):
    v = dataset[f] if f in ("Mean", "Min") else np.log10(dataset[f] + 1e-3)   # log pour les grandeurs positives
    sns.boxplot(x=dataset["Class"], y=v, ax=ax, showfliers=False,
                palette={0: "tab:blue", 1: "tab:red"}, hue=dataset["Class"], legend=False)
    ax.set_title(f if f in ("Mean", "Min") else f"log10({f})"); ax.set_xlabel(""); ax.set_ylabel("")
plt.suptitle("Distribution des features par classe (0 = hors crise, 1 = crise)", y=1.02)
plt.tight_layout(); plt.show()

✅ **Réponse Q10.**
- ⚠️ Les **moyennes** sont trompeuses ici : quelques fenêtres d'**artefacts** énormes (mouvements, électrode qui sature) gonflent la moyenne de la classe 0. C'est pour cela qu'on regarde aussi la **médiane**, plus robuste.
- En médiane, l'**écart-type**, la **RMS** et l'**amplitude** sont environ **5 fois plus élevés** en crise (Std : 23 → 126 µV). L'**énergie** et la **variance** (qui sont des carrés) sont **30 fois plus élevées**.
- La **puissance Delta** est **~40 fois plus élevée** en crise : c'est la plus forte augmentation, liée aux pointes-ondes à 3 Hz. Theta (×15), Alpha (×28) et Beta (×35) augmentent aussi, car la **pointe** est un événement brusque qui contient des hautes fréquences (harmoniques à 6, 9, 12 Hz…).
- La **moyenne du signal** (feature `Mean`) vaut ~0 dans les deux classes, car le filtrage centre le signal. Elle n'est **pas utile**.
- Les boîtes se chevauchent : certaines fenêtres hors crise très amples (artefacts, hyperpnée, somnolence) ressemblent à des crises. Ce seront les futurs **faux positifs**.

## Question 11 : matrice de corrélation

In [ ]:
corr = dataset[FEATURES + ["Class"]].corr(method="spearman")
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Matrice de corrélation (Spearman) entre features et classe"); plt.show()
corr["Class"].drop("Class").sort_values(ascending=False).round(3)

🧠 On utilise la corrélation de **Spearman** (sur les rangs) plutôt que Pearson, car les features de puissance ont des distributions très asymétriques avec des valeurs extrêmes.

✅ **Réponses Q11.**
1. **Fortement corrélées :** Std, Variance, RMS et Energy (corrélation ≈ 1). C'est logique : Variance = Std², et après filtrage (moyenne ≈ 0) RMS ≈ Std et Energy = N·RMS². Amplitude, Max et −Min sont aussi très liés à ces dernières. Les puissances par bande sont corrélées entre elles et avec l'énergie, car la somme des bandes ≈ l'énergie totale.
2. **Oui, il y a de la redondance :** Std, Variance, RMS et Energy apportent quasiment la **même information** (une seule suffirait). Ce n'est pas grave pour un arbre ou une forêt aléatoire, mais cela peut gêner le KNN, où la même information est comptée plusieurs fois dans la distance.
3. **Les plus discriminantes :** prises une par une, toutes les features d'amplitude et de puissance ont une corrélation similaire avec `Class` (≈ 0.29 en valeur absolue), tandis que **Mean** n'apporte rien (≈ 0). L'**importance des features de la Random Forest** (Partie IX) est plus parlante : **P_Beta** et **P_Alpha** arrivent en tête, devant l'Amplitude et P_Theta. **P_Delta** arrive presque en dernier. Pourquoi ? Beaucoup d'**artefacts** (mouvements des yeux, hyperpnée) produisent aussi de la puissance Delta, mais **pas** de hautes fréquences. Ce sont donc les « pointes » des pointes-ondes, visibles en Alpha et en Beta, qui permettent de distinguer une crise d'un artefact lent.

# Partie VIII : Construction des modèles

🧠 **Comprendre.**
- **Split 80/20 stratifié :** on garde 20 % des fenêtres de côté pour le test. *Stratifié* signifie que la proportion de crises (~3 %) est la même dans le train et dans le test. Sans cela, le test pourrait contenir trop peu de crises.
- **Normalisation (StandardScaler) :** KNN et SVM reposent sur des **distances**. Comme Energy vaut ~10⁷ et Mean ~0, l'énergie écraserait tout le reste. On ramène donc chaque feature à moyenne 0 et écart-type 1. On applique aussi un `log` aux features de puissance (Variance, Energy, P_*), car elles varient sur plusieurs ordres de grandeur. Les arbres, eux, n'en ont pas besoin.
- Le scaler est **appris sur le train uniquement**, grâce à `make_pipeline`, pour ne pas « tricher » avec les données de test.

In [ ]:
LOG_FEATS = ["Variance", "Energy", "P_Delta", "P_Theta", "P_Alpha", "P_Beta"]
X_all = dataset[FEATURES].copy()
X_all[LOG_FEATS] = np.log10(X_all[LOG_FEATS] + 1e-6)
y_all = dataset["Class"].values

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_all, y_all, dataset.index, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE)

print(f"Train : {len(X_train)} fenêtres ({100*y_train.mean():.2f} % de crises)")
print(f"Test  : {len(X_test)} fenêtres ({100*y_test.mean():.2f} % de crises)")

## Question 12 : quatre modèles

| Modèle | Principe |
|---|---|
| **Decision Tree** | une suite de questions « feature > seuil ? » apprises sur les données ; très lisible |
| **Random Forest** | des centaines d'arbres, chacun appris sur un sous-échantillon ; ils votent, ce qui est plus robuste qu'un arbre seul |
| **KNN** | on regarde les *k* fenêtres d'entraînement les plus proches et on vote |
| **SVM** | cherche la frontière qui sépare les classes avec la plus grande marge (noyau RBF : frontière courbe) |

In [ ]:
modeles = {
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    "KNN":           make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "SVM":           make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1, gamma="scale")),
}

resultats, predictions = [], {}
for nom, m in modeles.items():
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    predictions[nom] = y_pred
    resultats.append({"Modèle": nom,
                      "Accuracy":  accuracy_score(y_test, y_pred),
                      "Precision": precision_score(y_test, y_pred, zero_division=0),
                      "Recall":    recall_score(y_test, y_pred),
                      "F1-score":  f1_score(y_test, y_pred)})
res = pd.DataFrame(resultats).set_index("Modèle")
res.round(3)

# Partie IX : Évaluation des modèles

🧠 **Rappel des métriques** (la classe positive est **crise = 1**) :

|  | prédit 0 | prédit 1 |
|---|---|---|
| **réel 0** | VN (vrai négatif) | **FP** (fausse alarme) |
| **réel 1** | **FN** (crise ratée) | VP (crise détectée) |

- **Accuracy** = (VP+VN) / total : le pourcentage global de bonnes réponses ;
- **Precision** = VP / (VP+FP) : « quand le modèle dit crise, a-t-il raison ? » ;
- **Recall** (sensibilité) = VP / (VP+FN) : « quelle part des crises est détectée ? » ;
- **F1** = 2·P·R / (P+R) : la moyenne harmonique, qui n'est bonne que si P **et** R sont bonnes.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (nom, y_pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=["0 : pas crise", "1 : crise"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(nom)
plt.tight_layout(); plt.show()

res.plot(kind="bar", figsize=(10, 4), ylim=(0, 1.05), rot=0, title="Comparaison des modèles (jeu de test)")
plt.legend(loc="lower right"); plt.show()

In [ ]:
# Baseline "paresseuse" : toujours prédire 0
y_naif = np.zeros_like(y_test)
print(f"Modèle naïf (toujours 'pas de crise') : Accuracy = {accuracy_score(y_test, y_naif):.3f}, "
      f"Recall = {recall_score(y_test, y_naif):.3f}, F1 = {f1_score(y_test, y_naif):.3f}")

✅ **Interprétation des matrices de confusion.**
| Modèle | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| Decision Tree | 0.975 | 0.709 | 0.665 | 0.687 |
| **Random Forest** | **0.979** | **0.778** | **0.682** | **0.727** |
| KNN | 0.976 | 0.721 | 0.682 | 0.701 |
| SVM | 0.976 | 0.708 | 0.682 | 0.695 |

- Tous les modèles ont une **accuracy ≈ 97.5–98 %**, mais le modèle naïf fait déjà **96 %** avec un Recall de 0 ! L'accuracy ne permet donc **pas** de les départager.
- La différence se voit sur la **ligne « réel 1 »** des matrices de confusion : environ **2 fenêtres de crise sur 3 sont détectées** (Recall ≈ 0.68), et environ **1 alerte sur 4 est fausse** (Precision ≈ 0.71–0.78).
- La **Random Forest** est la meilleure (F1 = 0.727, meilleure Precision à Recall égal). C'est attendu : en combinant des centaines d'arbres, elle est plus robuste qu'un arbre seul et elle gère bien les features redondantes et les échelles différentes.
- Le **Decision Tree** seul est le moins bon (il sur-apprend). KNN et SVM sont proches.

# Question 13 : quelle métrique privilégier ?

✅ **Pourquoi l'accuracy seule est trompeuse.** Avec ~97 % de fenêtres « pas de crise », un modèle qui ne détecte **aucune** crise obtient ~97 % d'accuracy. L'accuracy est dominée par la classe majoritaire et ne dit rien sur la capacité à trouver les crises, qui est pourtant **le but du projet**.

**Métriques les plus pertinentes :**
- **Recall (sensibilité)**, la plus importante en médecine : une crise **ratée** (FN) est plus grave qu'une **fausse alarme** (FP), que le neurologue peut vérifier en quelques secondes. Pour un dispositif de surveillance, on veut rater le moins de crises possible.
- **Precision** : elle reste importante, car trop de fausses alarmes rendent le système inutilisable (« fatigue d'alarme »).
- **F1-score** : c'est le meilleur **résumé unique** pour comparer les modèles, car il combine Recall et Precision et n'est pas trompé par le déséquilibre.

👉 On choisit donc le meilleur modèle sur le **F1**, en surveillant particulièrement le **Recall**.

In [ ]:
MEILLEUR = res["F1-score"].idxmax()
print("Meilleur modèle (F1) :", MEILLEUR)
res.loc[[MEILLEUR]].round(3)

In [ ]:
# Importance des features selon la Random Forest
imp = pd.Series(modeles["Random Forest"].feature_importances_, index=FEATURES).sort_values()
imp.plot(kind="barh", figsize=(7, 4), title="Importance des features (Random Forest)"); plt.show()

# Partie X : Détection temporelle des crises

🧠 **Comprendre.** Le modèle prédit une classe par fenêtre. Pour retrouver des **crises** (début → fin), on :
1. trie les fenêtres d'un enregistrement dans l'ordre du temps ;
2. **fusionne** les fenêtres consécutives prédites « 1 » en un seul intervalle ;
3. *(post-traitement facultatif)* supprime les détections isolées trop courtes et recolle deux détections séparées par un trou de 1 s.

⚠️ **Choix méthodologique important.** Avec le split aléatoire de la Partie VIII, les fenêtres de test sont **éparpillées** dans le temps : on ne peut pas reconstruire une crise continue. De plus, les fenêtres voisines (qui se recouvrent à 50 %) se retrouvent les unes dans le train, les autres dans le test. C'est presque de la triche, et cela **surestime** les performances.

Pour une évaluation **réaliste**, on met donc de côté un **patient entier** que le modèle n'a jamais vu (**230718A-A**, 7 crises). On entraîne le meilleur modèle sur les 18 autres patients, puis on prédit tout l'enregistrement du patient mis de côté. C'est exactement la situation réelle : un nouveau patient arrive.

In [ ]:
PATIENT_TEST = "230718A-A"
masque = dataset.Patient == PATIENT_TEST

modele_final = {
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "SVM": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1, gamma="scale")),
}[MEILLEUR]
modele_final.fit(X_all[~masque], y_all[~masque])

d_test = dataset[masque].sort_values("t_debut").copy()
d_test["pred"] = modele_final.predict(X_all.loc[d_test.index])
print(f"{PATIENT_TEST} : {len(d_test)} fenêtres | F1 fenêtre = {f1_score(d_test.Class, d_test.pred):.3f}, "
      f"Recall = {recall_score(d_test.Class, d_test.pred):.3f}, Precision = {precision_score(d_test.Class, d_test.pred):.3f}")

## Question 14 : reconstruction des intervalles détectés

In [ ]:
def reconstruire_crises(t_debut, pred, win_s=WIN_S, step_s=WIN_S*(1-OVERLAP), duree_min=0, trou_max=0):
    # Fusionne les fenêtres consécutives prédites 1 en intervalles [début, fin] (en s)
    inter = []
    for t, p in zip(t_debut, pred):
        if p != 1: continue
        d, f = t, t + win_s
        if inter and d <= inter[-1][1] + trou_max:
            inter[-1][1] = max(inter[-1][1], f)
        else:
            inter.append([d, f])
    return [(d, f) for d, f in inter if f - d >= duree_min]

det_brut = reconstruire_crises(d_test.t_debut.values, d_test.pred.values)
det = reconstruire_crises(d_test.t_debut.values, d_test.pred.values, duree_min=3, trou_max=1)
print(f"Détections brutes : {len(det_brut)} | après post-traitement (durée ≥ 3 s, trous ≤ 1 s recollés) : {len(det)}\n")
for k, (d, f) in enumerate(det, 1):
    print(f"Crise détectée {k} : {d:7.1f} s -> {f:7.1f} s   (durée {f-d:4.1f} s)")
print()
for k, (d, f) in enumerate(CRISES[PATIENT_TEST], 1):
    print(f"Crise réelle   {k} : {d:7.1f} s -> {f:7.1f} s   (durée {f-d:4.1f} s)")

In [ ]:
def comparer(reelles, detectees):
    # Associe chaque crise réelle à une détection qui la chevauche
    lignes, utilisees = [], set()
    for k, (d, f) in enumerate(reelles, 1):
        match = [j for j, (a, b) in enumerate(detectees) if a < f and b > d]
        if match:
            j = match[0]; utilisees.update(match); a, b = detectees[j]
            lignes.append({"Crise réelle": k, "Début réel": d, "Fin réelle": f, "Début détecté": a, "Fin détectée": b,
                           "Écart début (s)": a - d, "Écart fin (s)": b - f, "Statut": "Détectée ✔"})
        else:
            lignes.append({"Crise réelle": k, "Début réel": d, "Fin réelle": f, "Statut": "RATÉE ✘ (FN)"})
    fp = [detectees[j] for j in range(len(detectees)) if j not in utilisees]
    return pd.DataFrame(lignes).round(1), fp

tab_cmp, fausses_alarmes = comparer(CRISES[PATIENT_TEST], det)
print("Fausses alarmes (détections sans crise réelle) :", [(round(a, 1), round(b, 1)) for a, b in fausses_alarmes])
tab_cmp

In [ ]:
Xp = SIGNAUX[PATIENT_TEST]
tp = np.arange(len(Xp)) / fs
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(tp, Xp[:, CANAUX_EEG.index("EEGF3")], lw=.3, color="k")
ylim = ax.get_ylim()
for i, (d, f) in enumerate(CRISES[PATIENT_TEST]):
    ax.axvspan(d, f, ymin=.5, ymax=1, color="tab:green", alpha=.4, label="crise réelle" if i == 0 else None)
for i, (d, f) in enumerate(det):
    ax.axvspan(d, f, ymin=0, ymax=.5, color="tab:red", alpha=.4, label="crise prédite" if i == 0 else None)
ax.set(title=f"{PATIENT_TEST} (patient jamais vu) : signal EEG (F3), crises réelles (haut) et prédites (bas)",
       xlabel="Temps (s)", ylabel="µV"); ax.legend(loc="upper right"); plt.show()

d0, f0 = CRISES[PATIENT_TEST][0]
plot_multicanal(Xp, fs, d0 - 10, f0 + 10, f"Zoom crise 1 : zone rouge = annotation réelle", crises=CRISES[PATIENT_TEST])

On regarde de plus près les deux erreurs : la fausse alarme vers 176–188 s et la crise ratée à 218–222 s.

In [ ]:
plot_multicanal(Xp, fs, 165, 230, "Zoom 165–230 s : rouge = crise annotée (218–222 s) ; détection du modèle = 176–188 s",
                crises=CRISES[PATIENT_TEST])

✅ **Réponse Q14.** Sur le patient **230718A-A**, que le modèle n'a jamais vu (7 crises annotées) :
- **6 crises sur 7 sont détectées**, avec un écart de **0 à 1 s** sur le début et la fin. La précision temporelle est limitée par le pas des fenêtres (1 s) ;
- **1 crise ratée** (crise n°2, 218–222 s) : elle ne dure que **4 s**, donc seulement 2–3 fenêtres, et son amplitude est plus faible ;
- **1 « fausse alarme »** de 12 s (176–188 s). Le zoom montre pourtant une **décharge de pointes-ondes rythmiques, ample, généralisée à tous les canaux**, exactement comme une absence. Il s'agit très probablement d'une **vraie crise oubliée dans l'Excel** : ici, c'est l'annotation qui est en défaut, pas le modèle. C'est une limite importante, car les erreurs d'annotation faussent aussi l'évaluation ;
- au niveau des fenêtres, on obtient F1 = 0.82, Recall = 0.80 et Precision = 0.84 ;
- le **post-traitement** (supprimer les détections < 3 s et recoller les trous de 1 s) fait passer de 8 à 7 détections, en supprimant une fausse alarme isolée.

# Partie XI : Analyse des erreurs

On analyse les erreurs du meilleur modèle sur le **jeu de test** (split 80/20).

In [ ]:
d_err = dataset.loc[idx_test].copy()
d_err["pred"] = predictions[MEILLEUR]
d_err["type"] = np.select(
    [(d_err.Class == 1) & (d_err.pred == 1), (d_err.Class == 0) & (d_err.pred == 0),
     (d_err.Class == 0) & (d_err.pred == 1), (d_err.Class == 1) & (d_err.pred == 0)],
    ["VP", "VN", "FP", "FN"], default="?")
print(d_err["type"].value_counts())

# Pour chaque erreur : quelle fraction de la fenêtre était réellement en crise ?
def frac_crise(r):
    a, b = r.t_debut, r.t_debut + WIN_S
    return sum(max(0, min(b, f) - max(a, d)) for d, f in CRISES[r.Patient]) / WIN_S
d_err["frac_crise"] = d_err.apply(frac_crise, axis=1)
d_err.groupby("type")[["frac_crise", "Energy", "P_Delta", "P_Alpha"]].median().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for typ, col in [("VN", "lightgray"), ("VP", "tab:green"), ("FP", "tab:orange"), ("FN", "tab:red")]:
    s = d_err[d_err.type == typ]
    ax.scatter(np.log10(s.P_Delta), np.log10(s.P_Alpha), s=8 if typ == "VN" else 18, c=col, label=f"{typ} ({len(s)})", alpha=.6)
ax.set(xlabel="log10 Puissance Delta", ylabel="log10 Puissance Alpha", title=f"Erreurs du modèle {MEILLEUR}")
ax.legend(); plt.show()

✅ **Analyse des erreurs.**
Sur les 5 994 fenêtres de test : **VN = 5 705**, **VP = 165**, **FN = 77**, **FP = 47**.
- **Vrais négatifs (VN)** : l'immense majorité ; l'activité de fond normale (énergie médiane ≈ 0.3·10⁶) est bien reconnue.
- **Vrais positifs (VP)** : des fenêtres de crise **très amples** (énergie médiane ≈ 11·10⁶), avec une forte puissance Delta **et** Alpha (les pointes).
- **Faux négatifs (FN)** : ce sont surtout des fenêtres de crise de **faible amplitude** (énergie médiane ≈ 2.7·10⁶, soit 4 fois moins que les VP). Ce sont des crises **plus discrètes**, chez certains patients ou en début et fin de crise. Elles ressemblent à de l'activité normale.
- **Faux positifs (FP)** : des fenêtres hors crise mais **amples en basse fréquence** (P_Delta proche de celle des VP), avec en revanche **moins d'Alpha** (moins de « pointes »). Ce sont souvent des **artefacts** (mouvements, clignements des yeux), des moments d'**hyperpnée** (HPN, qui ralentit l'EEG de l'enfant) ou de **somnolence**, ou encore des **bouffées de pointes-ondes réelles mais non annotées** dans l'Excel (par exemple pour 210427B-C, où le technicien a noté 3 absences alors que l'Excel n'en contient qu'une).

Sur le nuage de points, les FP (orange) se mélangent aux VP (vert), et les FN (rouge) se trouvent dans la zone des VN : la frontière entre les classes n'est pas nette.

# Partie XII : Amélioration du système

On teste trois améliorations et on mesure leur effet **sur le F1 de la classe crise**.

### Amélioration 1 : traiter le déséquilibre des classes (`class_weight="balanced"`)
🧠 On donne plus de poids aux erreurs sur la classe rare, ce qui pousse le modèle à moins rater de crises (Recall ↑).

In [ ]:
def evaluer(m, Xtr=X_train, Xte=X_test):
    m.fit(Xtr, y_train); p = m.predict(Xte)
    return {"Precision": precision_score(y_test, p), "Recall": recall_score(y_test, p), "F1": f1_score(y_test, p)}

amelio = {}
amelio["RF (base)"] = evaluer(RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE))
amelio["RF balanced"] = evaluer(RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                                                        min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE))
amelio["SVM balanced"] = evaluer(make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1, class_weight="balanced")))
pd.DataFrame(amelio).T.round(3)

### Amélioration 2 : nouvelles caractéristiques
🧠 Des features plus « intelligentes » pour les pointes-ondes :
- **puissances relatives** (Delta / puissance totale…) : elles sont indépendantes de l'amplitude propre à chaque patient ;
- la **fréquence dominante** et la **puissance dans la bande 2.5–4.5 Hz**, qui est la signature même de l'absence ;
- la **line length** (somme des |x[i+1] − x[i]|), un classique de la détection de crises ;
- la **synchronie** entre canaux (corrélation moyenne), car l'absence est généralisée et tous les canaux bougent ensemble.

In [ ]:
def features_avancees(fen, fs):
    spec = np.abs(np.fft.rfft(fen * HANN[None, :, None], axis=1))**2 / WIN
    f = np.fft.rfftfreq(fen.shape[1], 1/fs)
    tot = spec[:, (f >= 0.5) & (f < 30), :].sum(1) + 1e-9
    F = {}
    for nom, (lo, hi) in BANDES.items():
        F["R_" + nom] = spec[:, (f >= lo) & (f < hi), :].sum(1) / tot
    F["P_SW_2.5-4.5"] = np.log10(spec[:, (f >= 2.5) & (f <= 4.5), :].sum(1) + 1e-6)
    sl = (f >= 0.5) & (f < 30)
    F["Freq_dominante"] = f[sl][spec[:, sl, :].argmax(1)]
    F["LineLength"] = np.abs(np.diff(fen, axis=1)).sum(1)
    out = pd.DataFrame({k: v.mean(1) for k, v in F.items()})
    # synchronie : corrélation moyenne entre canaux
    z = (fen - fen.mean(1, keepdims=True)) / (fen.std(1, keepdims=True) + 1e-9)
    C = np.einsum("nti,ntj->nij", z, z) / fen.shape[1]
    k = fen.shape[2]
    out["Synchronie"] = (C.sum((1, 2)) - k) / (k * (k - 1))
    return out

lignes2 = []
for p, Xp_ in SIGNAUX.items():
    fen_p, _, _ = segmenter(Xp_, vecteur_classe(len(Xp_), fs, CRISES[p]), fs)
    lignes2.append(features_avancees(fen_p, fs))
feat2 = pd.concat(lignes2, ignore_index=True)
X_all2 = pd.concat([X_all.reset_index(drop=True), feat2], axis=1)
X_train2, X_test2 = X_all2.loc[idx_train], X_all2.loc[idx_test]

amelio["RF + nouvelles features"] = evaluer(RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE), X_train2, X_test2)
amelio["RF balanced + nouvelles features"] = evaluer(RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                                    min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE), X_train2, X_test2)
pd.DataFrame(amelio).T.round(3)

In [ ]:
rf2 = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE).fit(X_train2, y_train)
pd.Series(rf2.feature_importances_, index=X_all2.columns).sort_values().plot(
    kind="barh", figsize=(7, 6), title="Importance des features (avec nouvelles features)"); plt.show()

### Amélioration 3 : évaluation par patient (Leave-One-Patient-Out)
🧠 Comme expliqué en Partie X, le split aléatoire est **optimiste**, car des fenêtres qui se recouvrent se retrouvent dans le train et dans le test. Une évaluation honnête consiste à **tester chaque patient avec un modèle entraîné sur tous les autres** (`GroupKFold`). C'est plus dur, mais c'est ce qui compte en vrai.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_predict
groupes = dataset["Patient"].values
cv = GroupKFold(n_splits=len(np.unique(groupes)))
lopo = {}
for nom, Xc, m in [
    ("RF base (12 features)", X_all, RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=RANDOM_STATE)),
    ("RF balanced + nouvelles features", X_all2, RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample",
                                                     min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE))]:
    p = cross_val_predict(m, Xc, y_all, groups=groupes, cv=cv)
    lopo[nom] = {"Precision": precision_score(y_all, p), "Recall": recall_score(y_all, p), "F1": f1_score(y_all, p)}
pd.DataFrame(lopo).T.round(3)

✅ **Conclusion de la Partie XII.**
| Version | Precision | Recall | F1 |
|---|---|---|---|
| RF de base | 0.778 | 0.682 | 0.727 |
| RF balanced | 0.709 | 0.736 | 0.722 |
| SVM balanced | 0.392 | **0.847** | 0.536 |
| RF + nouvelles features | **0.828** | 0.698 | 0.758 |
| **RF balanced + nouvelles features** | 0.776 | 0.785 | **0.780** |

1. **Rééquilibrer les classes** (`class_weight`) fait monter le **Recall** (0.68 → 0.74 pour la RF), au prix d'un peu de Precision. Avec le SVM, l'effet est extrême : il détecte 85 % des crises mais déclenche beaucoup de fausses alarmes. C'est un **compromis** à régler selon l'usage clinique.
2. Les **nouvelles features** (puissances relatives, bande 2.5–4.5 Hz, line length, synchronie) font monter le F1 de 0.727 à **0.758**. Elles apparaissent parmi les plus importantes du modèle.
3. En **combinant les deux**, on obtient le meilleur résultat : **F1 = 0.780**, avec Precision et Recall équilibrés (≈ 0.78).
4. **Évaluation par patient :** le F1 passe de **0.704** (modèle de base) à **0.759** (modèle amélioré). C'est un peu plus bas que le split aléatoire (0.727 et 0.780) : c'est la performance **réaliste** attendue sur un nouveau patient. L'amélioration se confirme donc dans ce cadre plus strict.

**Autres pistes :** ajuster la taille de fenêtre (1 s pour plus de précision temporelle, 4 s pour plus de stabilité), optimiser les hyperparamètres (`GridSearchCV`), garder les features **par canal** au lieu de la moyenne, ajouter un lissage temporel des prédictions, ou utiliser le **deep learning** (CNN 1D sur le signal brut).

---
# Bilan et limites
- **Annotations :** l'heure de début des enregistrements n'était pas fournie. Elle a été estimée par recalage sur le signal, puis validée par les notes du technicien. Certaines crises réelles ne sont pas annotées dans l'Excel, et un patient avait une annotation erronée.
- **Peu de patients (20)** et peu de crises (92) : les résultats peuvent varier d'un patient à l'autre.
- **Fuite de données :** un split aléatoire de fenêtres qui se recouvrent surestime la performance. L'évaluation par patient est plus honnête.
- **Artefacts** (mouvements, hyperpnée, sommeil) : ce sont la principale source de fausses alarmes.
- **Résolution temporelle** limitée par la taille des fenêtres (±1–2 s).